In [1]:
#!/usr/bin/env python
import copy
import coffea
import numpy as np
import awkward as ak
import json
import hist
import yaml

# from mt2 import mt2

from coffea import processor
from coffea.analysis_tools import PackedSelection
from coffea.nanoevents.methods import vector
from coffea.lumi_tools import LumiMask

# silence warnings due to using NanoGEN instead of full NanoAOD
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema

from topcoffea.modules.paths import topcoffea_path
from topcoffea.modules.histEFT import HistEFT
import topcoffea.modules.eft_helper as efth
import topcoffea.modules.corrections as tc_cor
import topcoffea.modules.event_selection as tc_es

from ttbarEFT.modules.paths import ttbarEFT_path
from ttbarEFT.modules.analysis_tools import make_mt2, get_lumi
import ttbarEFT.modules.object_selection as tt_os
import ttbarEFT.modules.event_selection as tt_es
import ttbarEFT.modules.corrections as tt_cor 
from ttbarEFT.modules.processor_tools import calc_eft_weights
from ttbarEFT.modules.processor_tools import get_syst_lists

from topcoffea.modules.get_param_from_jsons import GetParam

get_tc_param = GetParam(topcoffea_path("params/params.json"))
get_tt_param = GetParam(ttbarEFT_path("params/params.json"))

NanoAODSchema.warn_missing_crossrefs = False
np.seterr(divide='ignore', invalid='ignore', over='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [2]:
fname = "/cms/cephfs/data/store/mc/RunIISummer20UL17NanoAODv9/TTTo2L2Nu_TuneCP5_13TeV-powheg-pythia8/NANOAODSIM/106X_mc2017_realistic_v9-v1/2510000/F689F2EB-6A85-CE46-9AB0-BF87D586AFFD.root" 
events = NanoEventsFactory.from_root(
    {fname: "Events"},
    schemaclass=NanoAODSchema,
    metadata={"dataset": "TTto2L2Nu"},
    mode="eager",
    entry_stop=5000,
).events()

In [4]:
year = "2017"
isData = False
dataset = "UL17_TTTo2L2Nu"
lep_cat = "em"
run_era = None
obj_correction_syst_lst = tt_cor.get_supported_jet_systematics(year, isData=isData, era=run_era)

In [6]:
######### Initialize Objects #########
met  = events.MET
ele  = events.Electron
mu   = events.Muon
tau  = events.Tau
jets = events.Jet 

In [7]:
leptonSelection = tt_os.Run2LeptonSelection()
######### Electron Selection ##########
ele['isGoodElec']=leptonSelection.is_sel_ele(ele)
ele_good = ele[ele.isGoodElec]

######### Muon Selection ##########
mu['pt'] = tt_cor.ApplyMuonPtCorr(mu, year, isData)
mu['isGoodMuon']=leptonSelection.is_sel_muon(mu)
mu_good = mu[mu.isGoodMuon]

leps = ak.concatenate([ele_good, mu_good], axis=1)
leps_sorted = leps[ak.argsort(leps.pt, axis=-1,ascending=False)] 

events['leps_pt_sorted'] = leps_sorted
tt_es.addLepCatMasks(events) 
tt_es.add2losMask(events, year, isData)

In [8]:
######### Jet Selections #########
jets['isClean'] = tt_os.isClean(jets, ele_good, drmin=0.4)& tt_os.isClean(jets, mu_good, drmin=0.4)
cleanedJets = jets[jets.isClean]
cleanedJets = ak.fill_none(cleanedJets, [])

In [10]:
# Medium DeepJet WP
medium_tag = "btag_wp_medium_" + year.replace("201", "UL1")
btagwpm = get_tc_param(medium_tag)


######### Add variables to EVENTS #########
events['leps_pt_sorted'] = leps_sorted
tt_es.addLepCatMasks(events) 
tt_es.add2losMask(events, year, isData)
tt_es.addmllMasks(events)

######## Create objects for dense axes ##########
leps_sorted = ak.pad_none(leps_sorted, 2)
l0 = leps_sorted[:,0]
l1 = leps_sorted[:,1]

ptll = (l0+l1).pt
mll = (l0+l1).mass

pass_trg = tt_es.trg_pass_no_overlap(events, isData, dataset, str(year), tt_es.triggers_dict, tt_es.exclude_triggers_dict, lep_cat)

In [11]:
btag_eff_lookup_m = tt_cor.GetBtagEffLookup(year, wp='medium')
light_btag_SF_lookup = tt_cor.GetBtagSFLookup(wp='M',year=year, method='deepJet_incl')
bc_btag_SF_lookup = tt_cor.GetBtagSFLookup(wp='M',year=year, method='deepJet_comb')

raw_met = met
cleanedJets['pt_orig'] = cleanedJets.pt     # NECESSARY FOR MET CORRECTIONS LATER 
rho_jagged = ak.ones_like(cleanedJets.pt) * events.fixedGridRhoFastjetAll
cleanedJets = ak.with_field(cleanedJets, rho_jagged, "Rho")
cleanedJets["pt_gen"] = ak.values_astype(ak.fill_none(cleanedJets.matched_gen.pt, 0), np.float32)
cleanedJets = tt_cor.ApplyJetCorrections(year, corr_type='jets', isData=isData, era=run_era).build(cleanedJets)
corrected_met = tt_cor.ApplyJetCorrections(year, corr_type='met', isData=isData, era=run_era).build(MET=raw_met, corrected_jets=cleanedJets)

In [26]:
print(f"orig pt: {raw_met.pt}")
print(f"corr pt: {corrected_met.pt}")

print(f"orig phi: {raw_met.phi}")
print(f"corr phi: {corrected_met.phi}")

orig pt: [38.7, 4.39, 138, 228, 76.3, 61.3, ..., 67.6, 98.6, 74.9, 8.85, 22.1, 55.2]
corr pt: [32, 7.3, 144, 219, 69.4, 60.8, 69.6, ..., 68.7, 80.1, 71.8, 9.75, 21.3, 52.5]
orig phi: [2.19, -1.77, 0.0761, -2.15, -2.88, 1.67, ..., 0.289, -2.76, 2.09, 0.214, 1.53]
corr phi: [2.31, -1.49, 0.115, -2.12, -2.88, 1.67, ..., 0.298, -2.65, 1.89, 0.257, 1.45]


In [29]:
for syst_var in obj_correction_syst_lst:
    correctedJets = tt_cor.ApplyJetSystematics(year=year, corr_type='jets', original_obj=cleanedJets, syst_var=syst_var)
    # met = tt_cor.ApplyJetCorrections(year, corr_type='met', isData=isData, era=run_era).build(MET=raw_met, corrected_jets=correctedJets)
    met = tt_cor.ApplyJetSystematics(year=year, corr_type='met', original_obj=corrected_met, syst_var=syst_var)
    
    print(f"systematic variation: {syst_var}")
    print(f"    Jet variables: ")
    print(f"        corr pt: {cleanedJets.pt}")
    print(f"        syst pt: {correctedJets.pt}")
    print(f"        corr mass: {cleanedJets.mass}")
    print(f"        syst mass: {correctedJets.mass}")
    
    print(f"    MET variables: ")
    print(f"        corr pt: {corrected_met.pt}")
    print(f"        syst pt: {met.pt}")
    print(f"        corr phi: {corrected_met.phi}")
    print(f"        syst phi: {met.phi}")

systematic variation: JER_2017Up
    Jet variables: 
        corr pt: [[121, 69.6, 52.5, 38.2, 33.4, 21.9], ..., [315, 149, 140, ..., 32.2, 15, 15.8]]
        syst pt: [[123, 70.1, 52.1, 38.3, 33.2, 22], ..., [317, 149, 140, ..., 32.2, 12, 15.9]]
        corr mass: [[10.8, 11, 6.67, 2.59, 5.99, 5.9], ..., [29.2, 21.3, 19.4, ..., 3.94, 3.72]]
        syst mass: [[11, 11.1, 6.63, 2.59, 5.97, 5.92], ..., [29.4, 21.4, 19.4, ..., 3.15, 3.75]]
    MET variables: 
        corr pt: [32, 7.3, 144, 219, 69.4, 60.8, 69.6, ..., 68.7, 80.1, 71.8, 9.75, 21.3, 52.5]
        syst pt: [31.3, 9.63, 145, 219, 67.9, 60.3, ..., 69.2, 76.8, 69.8, 10.4, 20.7, 48.9]
        corr phi: [2.31, -1.49, 0.115, -2.12, -2.88, 1.67, ..., 0.298, -2.65, 1.89, 0.257, 1.45]
        syst phi: [2.32, -1.3, 0.125, -2.12, -2.88, 1.66, ..., 0.3, -2.63, 1.82, 0.302, 1.4]
systematic variation: JER_2017Down
    Jet variables: 
        corr pt: [[121, 69.6, 52.5, 38.2, 33.4, 21.9], ..., [315, 149, 140, ..., 32.2, 15, 15.8]]
      